# Valuation-to-Option Pricing Lab

This notebook keeps two valuation tracks separate:

1. **Analyst-scenario discounted expected payoff:** five terminal equity-value scenarios with subjective weights.
2. **CRR risk-neutral lattice value:** a no-arbitrage option-pricing model built through repeated two-state replication.

The five analyst scenarios are not an expanded binomial tree and are not risk-neutral probabilities. They answer a different question: *Given these valuation beliefs and weights, what is the discounted expected option payoff?*

## 1. Load and validate student inputs

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd
ROOT=Path.cwd(); DATA=ROOT/'data'; DOCS=ROOT/'docs'; DOCS.mkdir(exist_ok=True)
inputs=pd.read_csv(DATA/'company_inputs.csv')
if len(inputs)!=1: raise ValueError('company_inputs.csv must contain exactly one firm row.')
M=inputs.iloc[0].to_dict()
for k in ['spot','strike','maturity_years','risk_free_rate','dividend_yield','volatility','market_bid','market_ask','dcf_value','comps_value']: M[k]=float(M[k])
M['lattice_steps']=int(M['lattice_steps']); M['market_mid']=(M['market_bid']+M['market_ask'])/2
M['option_type']=str(M['option_type']).lower(); M['exercise_style']=str(M['exercise_style']).lower()
if not 0<=M['risk_free_rate']<=1: raise ValueError('Enter risk_free_rate in decimal form, such as 0.045.')
if not 0<M['volatility']<=3: raise ValueError('Enter volatility in decimal form, such as 0.28.')
if M['market_bid']>M['market_ask']: raise ValueError('market_bid cannot exceed market_ask.')
if M['option_type'] not in ['call','put']: raise ValueError('option_type must be call or put.')
if M['exercise_style'] not in ['european','american']: raise ValueError('exercise_style must be european or american.')
targets=pd.read_csv(DATA/'analyst_targets.csv')
if len(targets)<5: raise ValueError('At least five analyst targets are required.')
if not targets.target.gt(0).all(): raise ValueError('Analyst targets must be positive.')
if targets.horizon_months.nunique()!=1: raise ValueError('All analyst targets must use one horizon.')
M

{'company': 'Asteron Systems',
 'ticker': 'ASTR',
 'valuation_date': '2026-10-16',
 'spot': 96.0,
 'strike': 105.0,
 'maturity_years': 1.0,
 'risk_free_rate': 0.045,
 'dividend_yield': 0.012,
 'volatility': 0.28,
 'volatility_type': 'implied',
 'option_type': 'call',
 'exercise_style': 'european',
 'lattice_steps': 200,
 'market_bid': 8.1,
 'market_ask': 8.6,
 'dcf_value': 106.0,
 'comps_value': 101.0,
 'market_mid': 8.35}

## 2. Equity valuation and analyst distribution

In [2]:
stats={'count':int(len(targets)),'mean':float(targets.target.mean()),'median':float(targets.target.median()),'std':float(targets.target.std(ddof=1)),'min':float(targets.target.min()),'max':float(targets.target.max())}
valuation={'Current market':M['spot'],'Comparable companies':M['comps_value'],'Student DCF':M['dcf_value'],'Analyst median':stats['median']}
pd.Series(valuation)

Current market           96.0
Comparable companies    101.0
Student DCF             106.0
Analyst median          101.0
dtype: float64

## 3. Five analyst-informed valuation states

The state prices are reproducible quantiles of the analyst-target distribution. The weights are disclosed subjective scenario masses, not risk-neutral probabilities.

- Severe bear: 5th percentile, 10% weight
- Bear: 25th percentile, 20% weight
- Base: 50th percentile, 40% weight
- Bull: 75th percentile, 20% weight
- Tail: 95th percentile, 10% weight

In [3]:
quantile_map={'Severe bear':.05,'Bear':.25,'Base':.50,'Bull':.75,'Tail':.95}
weights={'Severe bear':.10,'Bear':.20,'Base':.40,'Bull':.20,'Tail':.10}
states=pd.DataFrame({
    'state':list(quantile_map),
    'quantile':list(quantile_map.values()),
    'terminal_price':[float(targets.target.quantile(q)) for q in quantile_map.values()],
    'subjective_weight':[weights[k] for k in quantile_map]
})
assert np.isclose(states.subjective_weight.sum(),1.0)
states

,state,quantile,terminal_price,subjective_weight
0,Severe bear,0.05,86.2,0.1
1,Bear,0.25,95.0,0.2
2,Base,0.50,101.0,0.4
3,Bull,0.75,109.0,0.2
4,Tail,0.95,120.4,0.1


## 4. Analyst-scenario payoff decomposition

In [4]:
def payoff(stock,strike,kind):
    stock=np.asarray(stock,float)
    return np.maximum(stock-strike,0) if kind=='call' else np.maximum(strike-stock,0)
states['option_payoff']=payoff(states.terminal_price,M['strike'],M['option_type'])
states['weighted_payoff']=states.subjective_weight*states.option_payoff
weighted_terminal_payoff=float(states.weighted_payoff.sum())
discount_factor=float(np.exp(-M['risk_free_rate']*M['maturity_years']))
belief_value=discount_factor*weighted_terminal_payoff
weight_positive=float(states.loc[states.option_payoff>0,'subjective_weight'].sum())
weight_zero=1-weight_positive
print(f'Weight producing zero payoff: {weight_zero:.1%}')
print(f'Weight producing positive payoff: {weight_positive:.1%}')
print(f'Weighted terminal payoff: ${weighted_terminal_payoff:,.2f}')
print(f'Discount factor: {discount_factor:.4f}')
print(f'Analyst-scenario discounted expected payoff: ${belief_value:,.2f}')
states

Weight producing zero payoff: 70.0%
Weight producing positive payoff: 30.0%
Weighted terminal payoff: $2.34
Discount factor: 0.9560
Analyst-scenario discounted expected payoff: $2.24


,state,quantile,terminal_price,subjective_weight,option_payoff,weighted_payoff
0,Severe bear,0.05,86.2,0.1,0.0,0.00
1,Bear,0.25,95.0,0.2,0.0,0.00
2,Base,0.50,101.0,0.4,0.0,0.00
3,Bull,0.75,109.0,0.2,4.0,0.80
4,Tail,0.95,120.4,0.1,15.4,1.54


### Interpretation
A low analyst-scenario value is intuitive when much of the assigned weight ends below the call strike or above the put strike. The payoff contribution is visible state by state. This is not a replication price.

## 5. CRR risk-neutral lattice

In [5]:
def crr(spot,strike,T,r,sigma,N,q=0,kind='call',american=False):
    dt=T/N; u=np.exp(sigma*np.sqrt(dt)); d=1/u; growth=np.exp((r-q)*dt); p=(growth-d)/(u-d); disc=np.exp(-r*dt)
    if not(d<growth<u and 0<p<1): raise ValueError('No-arbitrage condition failed.')
    j=np.arange(N+1); stock=spot*u**j*d**(N-j); values=payoff(stock,strike,kind)
    for n in range(N-1,-1,-1):
        values=disc*(p*values[1:n+2]+(1-p)*values[:n+1])
        if american:
            j=np.arange(n+1); stock=spot*u**j*d**(n-j); values=np.maximum(values,payoff(stock,strike,kind))
    return {'price':float(values[0]),'u':float(u),'d':float(d),'p_star':float(p),'dt':float(dt)}
lattice=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],M['option_type'],M['exercise_style']=='american')
lattice

{'price': 8.332308324678896,
 'u': 1.0199962898353323,
 'd': 0.9803957229701683,
 'p_star': 0.49921736500371044,
 'dt': 0.005}

## 6. Theoretical validation and distribution comparison

In [6]:
pvs=M['spot']*np.exp(-M['dividend_yield']*M['maturity_years']); pvk=M['strike']*np.exp(-M['risk_free_rate']*M['maturity_years'])
lo,hi=(max(pvs-pvk,0),pvs) if M['option_type']=='call' else (max(pvk-pvs,0),pvk)
call=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'call')['price']
put=crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],M['lattice_steps'],M['dividend_yield'],'put')['price']
parity=float((call-put)-(pvs-pvk)); steps=[25,50,100,200,500]
conv=[crr(M['spot'],M['strike'],M['maturity_years'],M['risk_free_rate'],M['volatility'],n,M['dividend_yield'],M['option_type'])['price'] for n in steps]
analyst_log_dispersion=float(np.log(targets.target/M['spot']).std(ddof=1))
vol_horizon_dispersion=float(M['volatility']*np.sqrt(M['maturity_years']))
distribution_message='Analyst targets are narrower than the volatility input used by CRR.' if analyst_log_dispersion<vol_horizon_dispersion else 'Analyst targets are at least as dispersed as the volatility input used by CRR.'
validation={'valid_risk_neutral_probability':bool(0<lattice['p_star']<1),'inside_theoretical_bounds':bool(lo<=lattice['price']<=hi),'put_call_parity_pass':bool(abs(parity)<.05),'convergence_pass':bool(abs(conv[-1]-conv[-2])<.05),'parity_residual':parity,'convergence_gap':float(conv[-1]-conv[-2])}
assert all(validation[k] for k in ['valid_risk_neutral_probability','inside_theoretical_bounds','put_call_parity_pass','convergence_pass'])
print(distribution_message)
validation

Analyst targets are narrower than the volatility input used by CRR.


{'valid_risk_neutral_probability': True,
 'inside_theoretical_bounds': True,
 'put_call_parity_pass': True,
 'convergence_pass': True,
 'parity_residual': -4.796163466380676e-14,
 'convergence_gap': -0.0015202185624829667}

## 7. Export dashboard data

In [7]:
results={
 'model':M,'analyst_targets':[float(x) for x in targets.target],'analyst_stats':stats,'valuation':valuation,
 'states':states.to_dict('records'),'valuation_weighted_expected_payoff':belief_value,
 'belief_decomposition':{'weight_zero_payoff':weight_zero,'weight_positive_payoff':weight_positive,'weighted_terminal_payoff':weighted_terminal_payoff,'discount_factor':discount_factor},
 'distribution_comparison':{'analyst_log_dispersion':analyst_log_dispersion,'crr_horizon_volatility':vol_horizon_dispersion,'message':distribution_message},
 'lattice':lattice,'validation':validation,'convergence':[{'steps':n,'price':float(x)} for n,x in zip(steps,conv)],
 'comparison':{'Analyst-scenario discounted expected payoff':belief_value,'CRR risk-neutral lattice value':lattice['price'],'Observed market midpoint':M['market_mid']}
}
(DOCS/'model_results.json').write_text(json.dumps(results,indent=2))
print('Exported docs/model_results.json')
results['comparison']

Exported docs/model_results.json


{'Analyst-scenario discounted expected payoff': 2.237034107489453,
 'CRR risk-neutral lattice value': 8.332308324678896,
 'Observed market midpoint': 8.35}

## 8. Decision memo prompt

Explain why the analyst-scenario payoff and CRR value differ. Address the share of scenario weight producing a zero payoff, the contribution from positive-payoff states, the strike, analyst-target dispersion, the CRR volatility input, and why neither estimate should automatically be characterized as the uniquely correct market price.